In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip -q install -U ultralytics

In [ ]:
import os

DRIVE_ROOT = '/content/drive/MyDrive/bakalaura-darbs'
ZIP_PATH   = f'{DRIVE_ROOT}/yolo-dataset-split.zip'
DATA_ROOT  = '/content/yolo-dataset-split'

if not os.path.exists(DATA_ROOT):
    !cp "$ZIP_PATH" /content/
    !unzip -q /content/yolo-dataset-split.zip -d /content/

!ls $DATA_ROOT
!head -5 $DATA_ROOT/data.yaml

In [ ]:
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
MODELS = [
    ('YOLOv5n',  'yolov5n.pt'),
    ('YOLOv5s',  'yolov5s.pt'),
    ('YOLOv8n',  'yolov8n.pt'),
    ('YOLOv8s',  'yolov8s.pt'),
    ('YOLOv11n', 'yolo11n.pt'),
    ('YOLOv11s', 'yolo11s.pt'),
    ('YOLOv12n', 'yolo12n.pt'),
    ('YOLOv12s', 'yolo12s.pt'),
]

DATA_YAML = f'{DATA_ROOT}/data.yaml'
PROJECT   = f'{DRIVE_ROOT}/runs'

TRAIN_ARGS = dict(
    epochs=150,
    batch=16,
    imgsz=640,
    patience=30,
    seed=42,
    project=PROJECT,
    exist_ok=True,
    verbose=False,
)

In [ ]:
import time, traceback, glob
import pandas as pd
from ultralytics import YOLO

VAL_CSV   = f'{DRIVE_ROOT}/model_comparison.csv'
TEST_CSV  = f'{DRIVE_ROOT}/test_results.csv'
SPEED_CSV = f'{DRIVE_ROOT}/inference_speed.csv'

def load_or_empty(path, cols):
    if os.path.exists(path):
        return pd.read_csv(path).to_dict('records')
    return []

def save(rows, path):
    pd.DataFrame(rows).to_csv(path, index=False)
    print(f'    → {path}')

val_rows   = load_or_empty(VAL_CSV,   ['model'])
test_rows  = load_or_empty(TEST_CSV,  ['model'])
speed_rows = load_or_empty(SPEED_CSV, ['model'])
done_val   = {r['model'] for r in val_rows}
done_test  = {r['model'] for r in test_rows}
done_speed = {r['model'] for r in speed_rows}

sample_test_images = sorted(glob.glob(f'{DATA_ROOT}/images/test/*.png'))

for name, checkpoint in MODELS:
    print(f"\n{'='*70}\n{name}\n{'='*70}")
    run_dir = f'{PROJECT}/{name}'
    weights = f'{run_dir}/weights/best.pt'

    if os.path.exists(weights):
        print(f'  [train] best.pt exists, skipping training')
    else:
        try:
            t0 = time.time()
            YOLO(checkpoint).train(data=DATA_YAML, name=name, **TRAIN_ARGS)
            print(f'  [train] done in {(time.time()-t0)/60:.1f} min')
        except Exception as e:
            print(f'  [train] FAILED: {e}')
            traceback.print_exc()
            continue

    if name not in done_val:
        csv_path = f'{run_dir}/results.csv'
        if os.path.exists(csv_path):
            df = pd.read_csv(csv_path)
            df.columns = [c.strip() for c in df.columns]
            last = df.iloc[-1]
            val_rows.append({
                'model':     name,
                'mAP50':     round(last['metrics/mAP50(B)'], 4),
                'mAP50-95':  round(last['metrics/mAP50-95(B)'], 4),
                'precision': round(last['metrics/precision(B)'], 4),
                'recall':    round(last['metrics/recall(B)'], 4),
                'time_min':  round(last['time'] / 60, 1),
            })
            save(val_rows, VAL_CSV)
        else:
            print(f'  [val]  no results.csv — skipping')

    if name not in done_test and os.path.exists(weights):
        try:
            print(f'  [test] evaluating on held-out test set…')
            metrics = YOLO(weights).val(
                data=DATA_YAML, split='test',
                project=PROJECT, name=f'{name}_test', exist_ok=True,
            )
            test_rows.append({
                'model':          name,
                'test_mAP50':     round(metrics.box.map50, 4),
                'test_mAP50-95':  round(metrics.box.map, 4),
                'test_precision': round(metrics.box.mp, 4),
                'test_recall':    round(metrics.box.mr, 4),
            })
            save(test_rows, TEST_CSV)
        except Exception as e:
            print(f'  [test] FAILED: {e}')

    if name not in done_speed and os.path.exists(weights):
        try:
            model = YOLO(weights)
            model.predict(sample_test_images[0], verbose=False)
            t0 = time.time()
            for img in sample_test_images:
                model.predict(img, verbose=False)
            ms = (time.time() - t0) * 1000 / len(sample_test_images)
            speed_rows.append({'model': name, 'ms_per_image': round(ms, 1)})
            save(speed_rows, SPEED_CSV)
            print(f'  [speed] {ms:.1f} ms/image on {len(sample_test_images)} test images')
        except Exception as e:
            print(f'  [speed] FAILED: {e}')

print('\n=== ALL DONE ===')
print('Files in Drive:')
for p in (VAL_CSV, TEST_CSV, SPEED_CSV):
    print(f'  {p}  ({os.path.getsize(p)} bytes)' if os.path.exists(p) else f'  {p}  MISSING')
print(f'  Per-model run dirs under: {PROJECT}/')

In [ ]:
print('=== Validation metrics ===')
print(pd.read_csv(VAL_CSV).to_string(index=False) if os.path.exists(VAL_CSV) else 'missing')
print('\n=== Test-set metrics (these go into the thesis) ===')
print(pd.read_csv(TEST_CSV).to_string(index=False) if os.path.exists(TEST_CSV) else 'missing')
print('\n=== Inference speed (ms/image on current GPU) ===')
print(pd.read_csv(SPEED_CSV).to_string(index=False) if os.path.exists(SPEED_CSV) else 'missing')